# Simple E-NAS

### Notebook configuration

In [1]:
import os


notebook_path = os.getcwd() 
notebook_list_path = notebook_path.split('\\')

ann_list_path = []
for folder_index in range (len(notebook_list_path)-2):
    ann_list_path.append(notebook_list_path[folder_index])
    ann_list_path.append('\\')
ann_list_path.append('ANNs')

ann_path = ''.join(ann_list_path)

import sys
sys.path.append(ann_path)

## Problem statment
Finding the optimum number of neurons in hidden layers for the model that recognizes digits.

## Data
In this solution one individual is a list of lists. Each list represent one hidden layer, and each layer(list) contains 0 and 1 that represents bits that will give us number of neurons in each layer.
We will try to create data for 2 layers and maksimum 255 neurons.

Example:
```
individual = [
    [  # first layer
        0, 0, 0, 0, 0, 1, 0, 1  # bit representation of number of neural inputs
    ],
    [  # second layer
        0, 0, 0, 0, 0, 1, 0, 1  # bit representation of number of neural inputs
    ]
]
```

Note:

In number of layers we will not counting output layer due to it always need to have 10 outputs.
Allso we will not include input layer, due to it's depend on image (for dense layers).

In [2]:
seed_data = [
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
]

bits_in_number = 8

## GA initialization

In [3]:
from pyeasyga import pyeasyga

ga = pyeasyga.GeneticAlgorithm(seed_data,
                               population_size=15,
                               generations=100,
                               crossover_probability=0.8,
                               mutation_probability=0.05,
                               elitism=True,
                               maximise_fitness=True)

## Define Individual
An individual is an entity of data. COnception for library `pyeasyga` is that the build-in algorithm will create many variations of individuals and in this way, we receive datasets for genetic algorithm.

In [4]:
import random


# define and set function to create a candidate solution representation
def create_individual(data):
    individual = data[:]
    for layer_index in range(len(individual)):
        for bit_index in range(len(individual[layer_index])):
            individual[layer_index][bit_index] = random.randint(0, 1)

    return individual

In [5]:
ga.create_individual = create_individual

## Fitness function

We will use as fitness function:

`f(n1, n2) = MSE(n1, n2) + alfa((n1 + n2)/2(2^N -1))*MSE(n1, n2), 0 < alfa < 1`

Where: 
* n1 - number of neurons in first layer 
* n2 - number of neurons in second layer
* MSE - Mean Squared Error
* 2(2^N-1) - maximum number of neurons in the networ
* N - the number of bits on which the number of neurons in the layer is coded

Reminder:

In number of layers we will not counting output layer due to it always need to have 10 outputs.
Allso we will not include input layer, due to it's depend on image (for dense layers).

In [6]:
from cifar10_number_of_neurons import (prepare_cifar10_data, get_model)


x_train, y_train, x_test, y_test = prepare_cifar10_data()

def count_mse(first_layer_number_of_neurons, second_layer_number_of_neurons):

    model = get_model(
        first_layer_number_of_neurons, 
        second_layer_number_of_neurons, 
        x_train, 
        y_train
    )

    mse = 0
    for loss in model.history['loss']:
        mse += loss
    
    return mse

In [7]:
from utils import (individual_guard, convert_list_to_int)


def fitness(individual, data):
    fitness = 0

    if individual_guard(individual):  # We need this guard due to data corruption (look to notes)
        first_layer_number_of_neurons = convert_list_to_int(individual[0])
        second_layer_number_of_neurons = convert_list_to_int(individual[1])

        mse = count_mse(first_layer_number_of_neurons, second_layer_number_of_neurons)

        alfa = 0.2
        total_number_of_neurons = first_layer_number_of_neurons + second_layer_number_of_neurons
        maximum_nuerons_in_layer = 2^bits_in_number
        total_maximum_neurons = 2*(maximum_nuerons_in_layer - 1)
        
        punishment_for_large_network_complexity = alfa * (total_number_of_neurons/total_maximum_neurons) * mse
        
        fitness = mse + punishment_for_large_network_complexity

    return fitness


In [8]:
ga.fitness_function = fitness

In [11]:
import tensorflow_datasets as tfds
import tensorflow as tf


def normalize_img(image, label):
        """Normalizes images: `uint8` -> `float32`."""
        return tf.cast(image, tf.float32) / 255., label

def prepare_ann_data():
    (ds_train, ds_test), ds_info = tfds.load(
        'mnist',
        split=['train', 'test'],
        shuffle_files=True,
        as_supervised=True,
        with_info=True,
    )

    ds_train = ds_train.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds_train = ds_train.cache()
    ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
    ds_train = ds_train.batch(128)
    ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

    ds_test = ds_test.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds_test = ds_test.batch(128)
    ds_test = ds_test.cache()
    ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

    return ds_train, ds_test

train, test = prepare_ann_data()

In [12]:
train[0]

<PrefetchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

## Run GA

In [9]:
ga.run()

Epoch 1/50


ValueError: in user code:

    File "c:\Users\jedrz\anaconda3\lib\site-packages\keras\engine\training.py", line 1051, in train_function  *
        return step_function(self, iterator)
    File "c:\Users\jedrz\anaconda3\lib\site-packages\keras\engine\training.py", line 1040, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\jedrz\anaconda3\lib\site-packages\keras\engine\training.py", line 1030, in run_step  **
        outputs = model.train_step(data)
    File "c:\Users\jedrz\anaconda3\lib\site-packages\keras\engine\training.py", line 894, in train_step
        return self.compute_metrics(x, y, y_pred, sample_weight)
    File "c:\Users\jedrz\anaconda3\lib\site-packages\keras\engine\training.py", line 987, in compute_metrics
        self.compiled_metrics.update_state(y, y_pred, sample_weight)
    File "c:\Users\jedrz\anaconda3\lib\site-packages\keras\engine\compile_utils.py", line 501, in update_state
        metric_obj.update_

## Results

In [ ]:
result = ga.best_individual()
print(result)

neurons_in_layers = [convert_list_to_int(result[1][0]), convert_list_to_int(result[1][1])]

print(neurons_in_layers)

We may receive (well, I received them, when last time I run it) values 137 for the first layer and 77 for the second.

As you can see, you can use genetic algorithms to determine the hyperparameters of the neural network.

This is only a simple example, and further research is needed to determine whether this method is useful for more complex networks.

Main bottle necks:
* execution time - training each model is taking too much time
* fitness function - one used in this example is very simple and covers an only number of neurons. To be fully useful we should adjust a number of neurons, layers, type of layers, layers organization, and so on.

## Notes
Some data appear to have structure like `[[...], 0]` (`list[list, int]`) and not `[[...], [...]]` (`list[list, list]`). 

Individuals are generated correctly but thic corrupted data are passed to fitness function, so error is somwhere between.

For noe I do not know what causes this error.